
## Ejercicio 3

Para resolver este ejercicio utilice un modelo de red neuronal convolucional que reconozca la cantidad de dedos extendidos en cada mano de las imágenes del dataset **“Fingers”**.

<div style="text-align: center;">
    <img src='../imagenes/p4-ej10.png' width="40%">
</div>

La versión original de este de estas imágenes se encuentra en https://www.kaggle.com/koryakinp/fingers.

Puede hallar una versión reducida de estas imágenes en el Moodle del curso, en la misma sección donde se encuentra este enunciado de práctica. También encontrará allí ejemplos sobre cómo cargar estas imágenes y cómo procesarlas con una red neuronal convolucional.

### a)

Entrene y pruebe un modelo utilizando los datos de las carpetas **test** y **train**, midiendo **accuracy**.

In [1]:
FUENTES_DIR = '../Fuentes'         # carpeta donde se encuentran archivos .py auxiliares
DATOS_DIR   = '../Data_Sets/p5/' # carpeta donde se encuentran los datasets
LOCAL_DIR = DATOS_DIR
import sys
sys.path.append(FUENTES_DIR)

### Preparación de Dataset

Para poder trabajar con el dataset se debe:
1. Copiar el archivo Fingers.zip en la carpeta drive de los datos (establecida para este script en DATOS_DIR).
2. Descomprimir con Unzip los archivos

In [3]:
import zipfile
import os

# Descomprimir en la misma carpeta
with zipfile.ZipFile(os.path.join(DATOS_DIR, 'Fingers.zip'), 'r') as zip_ref:
    zip_ref.extractall(LOCAL_DIR)

# por si se nececita borrar la carpeta, descomentar
#!rm -r Fingers/DATOS

### Carga datos de entrenamiento y prueba

En esta sección se cargan todas las imagenes que se encuentran en subcarpetas.
Se obtiene un listado de todos los archivos (**glob.glob**) y a partir de este se cargan las imágenes, se escalan para que los niveles queden entre 0 y 1 y se agregan a una lista. También se extrae del nombre la clase a la que pertenece la imagen (la cantidad de dedos de la mano es parte del nombre del archivo xxxx_1L.png).
Crea tres subconjuntos de datos: entrenamiento, validación y prueba.

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input, Conv2D, MaxPooling2D, LeakyReLU
from tensorflow.keras import optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from PIL import Image
import numpy as np
import glob
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IMG_ERROR = 'No hay imágenes para cargar. Verificar que la ruta sea correcta y que la carpeta tenga imagenes con la extensión usada'

def import_data(data_dir):

    img_list = glob.glob(data_dir)      # Obtener la lista de archivos de imágenes

    assert len(img_list) > 0, IMG_ERROR # verifica que la ruta sea correcta y tenga al menos 1 imagen

    img_data = []  # lista de imagenes
    lbl_data = []  # lista de etiquetas

    img_count = len(img_list)
    for i, img_path in enumerate(img_list):

        img = Image.open(img_path)          # Carga imagen
        img = np.array(img) / np.max(img)   # Normaliza los píxeles entre 0 y 1
        img = img.reshape((*img.shape, 1))  # Formatea la imagen para TF: WxH => WxHx1

        # Almacenar la imagen y la etiqueta
        img_data.append(img)
        # ej. nombre de archivo: 000e7aa6-100b-4c6b-9ff0-e7a8e53e4465_5L.png
        lbl_data.append(int(img_path[-6]))  # Extrae la cantidad de dedos del nombre del archivo

        # Mostrar progreso en la carga
        if i % 100 == 0:
            print("\rCargando imágenes: %6.2f%%" % (100 * i / img_count), end="")

    print("\rCargando imágenes: 100.00%% (%d) \n" % img_count)

    return np.array(img_data), np.array(lbl_data)


# carga las imagenes a partir de los nombres de archivos
x_train, y_train = import_data(LOCAL_DIR+"Fingers/train/*/*.png")

# carga las imagenes a partir de los nombres de archivos
x_test, y_test = import_data(LOCAL_DIR+"Fingers/test/*/*.png")

# separa los datos y clase en grupo de entrenamiento y validacion
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = 0.30, shuffle = True)


2025-10-18 13:04:00.651995: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-18 13:04:00.652250: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-18 13:04:00.687153: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-18 13:04:01.699172: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Cargando imágenes: 100.00% (17999) 

Cargando imágenes: 100.00% (3600) 



### Genera y entrena el modelo

In [5]:
EPOCAS = 100
LOTES  = 128
PACIENCIA=5
IMG_SIZE = x_train.shape[1:]
N_CLASSES = len(np.unique(y_train))
ACTIVA =LeakyReLU()

# Construye el modelo
model = Sequential()

model.add(Input( shape=IMG_SIZE ))
model.add(Conv2D(16, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(32, activation = ACTIVA))
model.add(Dense(N_CLASSES, activation = 'softmax'))

model.summary()

# construye el modelo
optimizer = optimizers.Adam(0.001)
# Observar que con "sparse_categorical_crossentropy" no hace falta codificacion one-hot para las clases
model.compile(optimizer, loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])


# parada temprana para evitar el sobreajuste
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, min_delta=0.001, restore_best_weights=True )

# entrena el modelo y guarda la historia del progreso
H = model.fit(x_train,
              y_train,
              batch_size = LOTES,
              epochs = EPOCAS,
              validation_data = (x_val, y_val),
              callbacks=[early_stop]
             )

2025-10-18 13:04:12.261775: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       262,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 267,174 (1.02 MB)

 Trainable params: 267,174 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
 1/99 ━━━━━━━━━━━━━━━━━━━━ 1:25 873ms/step - accuracy: 0.1094 - loss: 1.7939

E0000 00:00:1760803453.305840   94216 meta_optimizer.cc:967] remapper failed: INVALID_ARGUMENT: Mutation::Apply error: fanout 'StatefulPartitionedCall/gradient_tape/sequential_1/conv2d_1/leaky_re_lu_1/LeakyRelu/LeakyReluGrad' exist for missing node 'StatefulPartitionedCall/sequential_1/conv2d_1/BiasAdd'.


99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - accuracy: 0.8925 - loss: 0.3666 - val_accuracy: 0.9893 - val_loss: 0.0449
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - accuracy: 0.9947 - loss: 0.0222 - val_accuracy: 0.9991 - val_loss: 0.0097
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 60ms/step - accuracy: 0.9990 - loss: 0.0068 - val_accuracy: 1.0000 - val_loss: 0.0045
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 71ms/step - accuracy: 0.9994 - loss: 0.0032 - val_accuracy: 0.9998 - val_loss: 0.0041
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 1.0000 - val_loss: 8.1547e-04
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 1.0000 - loss: 8.1023e-04 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 1.0000 - loss: 4.5969e-04 - val_accuracy: 1.0000 - val_loss: 3.7512e-04
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy: 1.0000 - loss: 2.8192e-04 - val_accu

### Guardado del modelo


In [6]:
model.save('Modelos/Fingers_3_conv_model.keras')

### Graficas y métricas de la evolución del modelo

In [7]:
# %% evalua el modelo para entrenamiento y testeo
pred_train = model.evaluate(x_train, y_train, verbose=0)
pred_test = model.evaluate(x_test, y_test, verbose=0)
print("\nEfectividad del modelo con datos de entrenamiento: %6.2f%%" % (H.history['accuracy'][-1]*100))
print("Efectividad del modelo con datos de validacion...: %6.2f%%" % (H.history['val_accuracy'][-1]*100))
print("Efectividad del modelo con datos de Prueba.......: %6.2f%%" % (pred_test[1]*100))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Pérdida", "Precisión"))

fig.add_trace(go.Scatter(y=H.history["loss"], mode='lines', name='train_loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=H.history["val_loss"], mode='lines', name='val_loss'), row=1, col=1)

fig.add_trace(go.Scatter(y=H.history["accuracy"], mode='lines', name='train_acc'), row=1, col=2)
fig.add_trace(go.Scatter(y=H.history["val_accuracy"], mode='lines', name='val_acc'), row=1, col=2)

fig.update_layout(
    title="Entrenamiento y Validación",
    xaxis_title="Época",
    yaxis_title="Valor",
    legend_title="Métricas",
    width=1200,
    height=600,
    template="plotly_white"  # Plantilla de diseño
)


fig.show()


Efectividad del modelo con datos de entrenamiento: 100.00%
Efectividad del modelo con datos de validacion...: 100.00%
Efectividad del modelo con datos de Prueba.......:  99.97%


### b)

Genere una versión del **dataset** para **test** agregando transformaciones al azar sobre imágenes originales. Haga **rotaciones** entre -45° y 45°, repita el test y mida el accuracy.

In [8]:
from random import random
from scipy.ndimage import rotate

def rotar_al_azar(data_imgs, max_ang):
    result = np.empty_like(data_imgs)
    for i, img in enumerate(data_imgs):
        ang = (random() - 0.5) * 2 * max_ang #random genera nros aleatorios entre 0 y 1
        result[i] = rotate(img, ang, reshape=False, mode='reflect')
    return result

x_test_rot = rotar_al_azar(x_test, 45)

# evalua el modelo con los datos de testeo
pred = model.evaluate(x_test_rot, y_test, verbose=0)
print("Efectividad del modelo con datos de Prueba.......: %6.2f%%" % (pred[1]*100))

# Muestra 10 de las imágenes rotadas
num_imagenes_a_mostrar = 10
imagenes_rotadas = x_test_rot[:num_imagenes_a_mostrar]

Efectividad del modelo con datos de Prueba.......:  71.53%


### c)

Genere una versión del **dataset train** como en b) y repita el entrenamiento y prueba del punto a) con los datasets modificados.

In [ ]:

from random import random
from scipy.ndimage import rotate
import numpy as np
from tensorflow.keras import Sequential, optimizers, callbacks
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, LeakyReLU

# --- Función auxiliar para rotar ---
def rotar_al_azar(data_imgs, max_ang):
    result = np.empty_like(data_imgs)
    for i, img in enumerate(data_imgs):
        ang = (random() - 0.5) * 2 * max_ang  # random genera nros entre 0 y 1
        result[i] = rotate(img, ang, reshape=False, mode='reflect')
    return result

# Aplica rotación al azar al conjunto de entrenamiento
x_train_rot = rotar_al_azar(x_train, 45)

# (Opcional) también podés rotar el conjunto de validación, aunque no es obligatorio
x_val_rot = rotar_al_azar(x_val, 45)

# --- Parámetros de entrenamiento ---
EPOCAS = 100
LOTES  = 128
PACIENCIA = 5
IMG_SIZE = x_train.shape[1:]
N_CLASSES = len(np.unique(y_train))
ACTIVA = LeakyReLU()

# --- Define el modelo ---
model_rot = Sequential()
model_rot.add(Input(shape=IMG_SIZE))
model_rot.add(Conv2D(16, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model_rot.add(MaxPooling2D(pool_size=(2,2)))
model_rot.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model_rot.add(MaxPooling2D(pool_size=(2,2)))
model_rot.add(Flatten())
model_rot.add(Dense(32, activation=ACTIVA))
model_rot.add(Dense(N_CLASSES, activation='softmax'))

model_rot.summary()

# --- Compila el modelo ---
optimizer = optimizers.Adam(0.001)
model_rot.compile(optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# --- Early stopping ---
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=PACIENCIA, min_delta=0.001, restore_best_weights=True
)

# --- Entrenamiento con datos rotados ---
H_rot = model_rot.fit(
    x_train_rot,
    y_train,
    batch_size=LOTES,
    epochs=EPOCAS,
    validation_data=(x_val_rot, y_val),
    callbacks=[early_stop],
    verbose=1
)

# --- Evaluación del modelo rotado ---
pred_rot = model_rot.evaluate(x_test, y_test, verbose=0)
print("Efectividad del modelo ENTRENADO CON IMÁGENES ROTADAS (test normal): %6.2f%%" % (pred_rot[1]*100))

# Evaluacion con test rotado para comparar:
x_test_rot = rotar_al_azar(x_test, 45)
pred_rot_test = model_rot.evaluate(x_test_rot, y_test, verbose=0)
print("Efectividad del modelo ENTRENADO CON IMÁGENES ROTADAS (test rotado): %6.2f%%" % (pred_rot_test[1]*100))


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │       262,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 267,174 (1.02 MB)

 Trainable params: 267,174 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
 3/99 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.1359 - loss: 1.7925  

E0000 00:00:1760804608.342236   94216 meta_optimizer.cc:967] remapper failed: INVALID_ARGUMENT: Mutation::Apply error: fanout 'StatefulPartitionedCall/gradient_tape/sequential_1_1/conv2d_3_1/leaky_re_lu_1_1/LeakyRelu/LeakyReluGrad' exist for missing node 'StatefulPartitionedCall/sequential_1_1/conv2d_3_1/BiasAdd'.


99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - accuracy: 0.7319 - loss: 0.7476 - val_accuracy: 0.9513 - val_loss: 0.2004
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - accuracy: 0.9712 - loss: 0.1097 - val_accuracy: 0.9902 - val_loss: 0.0570
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - accuracy: 0.9898 - loss: 0.0447 - val_accuracy: 0.9959 - val_loss: 0.0260
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy: 0.9964 - loss: 0.0208 - val_accuracy: 0.9963 - val_loss: 0.0194
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.9981 - loss: 0.0119 - val_accuracy: 0.9946 - val_loss: 0.0188
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.9986 - loss: 0.0088 - val_accuracy: 0.9983 - val_loss: 0.0100
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.9993 - loss: 0.0062 - val_accuracy: 0.9970 - val_loss: 0.0116
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.9987 - loss: 0.0069 - val_accuracy: 0.9985 - val_l